# 

In [23]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list

In [24]:
v2_test_path = "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_molpo-v2-ablation"
v2_train_path = "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_molpo-v2-ablation"

v2_test_data = datasets.load_from_disk(v2_test_path)
v2_train_data = datasets.load_from_disk(v2_train_path)

In [25]:
v2_test_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string', '0-th_rejected_selfies', '0-th_rejected_idx', '0-th_rejected_target_text', '0-th_rejected_x', '0-th_rejected_edge_index', '0-th_rejected_edge_attr', '0-th_additional_rejected_x', '0-th_additional_rejected_edge_index', '0-th_additional_rejected_edge_attr', '1-th_rejected_selfies', '1-th_rejected_idx', '1-th_rejected_target_text', '1-th_rejected_x', '1-th_rejected_edge_index', '1-th_rejected_edge_attr', '1-th_additional_rejected_x', '1-th_additional_rejected_edge_index', '1-th_additional_rejected_edge_attr', '2-th_rejected_selfies', '2-th_rejected_idx', '2-th_rejected_target_text', '2-th_rejected_x', '2-th_rejected_edge_index', '2-th_rejected_edge_attr', '2-th_additional_rejected_x', '2-th_additional_rejected_edge_index', '2-th_additional_rejected_edge_attr', '3-th_rejected_selfies', '3-th_rejected_idx', '3-th_

In [26]:
v2_train_data

Dataset({
    features: ['task', 'x', 'edge_index', 'edge_attr', 'additional_x', 'additional_edge_index', 'additional_edge_attr', 'prompt_text', 'target_text', 'input_mol_string', '0-th_rejected_selfies', '0-th_rejected_idx', '0-th_rejected_target_text', '0-th_rejected_x', '0-th_rejected_edge_index', '0-th_rejected_edge_attr', '0-th_additional_rejected_x', '0-th_additional_rejected_edge_index', '0-th_additional_rejected_edge_attr', '1-th_rejected_selfies', '1-th_rejected_idx', '1-th_rejected_target_text', '1-th_rejected_x', '1-th_rejected_edge_index', '1-th_rejected_edge_attr', '1-th_additional_rejected_x', '1-th_additional_rejected_edge_index', '1-th_additional_rejected_edge_attr', '2-th_rejected_selfies', '2-th_rejected_idx', '2-th_rejected_target_text', '2-th_rejected_x', '2-th_rejected_edge_index', '2-th_rejected_edge_attr', '2-th_additional_rejected_x', '2-th_additional_rejected_edge_index', '2-th_additional_rejected_edge_attr', '3-th_rejected_selfies', '3-th_rejected_idx', '3-th_

In [27]:
list_task = set(v2_test_data['task'])
list_task

{'bace',
 'chebi-20-mol2text',
 'forward_reaction_prediction',
 'qm9_homo',
 'reagent_prediction',
 'smol-property_prediction-bbbp',
 'smol-property_prediction-clintox',
 'smol-property_prediction-esol',
 'smol-property_prediction-hiv',
 'smol-property_prediction-lipo',
 'smol-property_prediction-sider'}

In [28]:
task_specific_test_data = {}
for task in list_task:
    task_specific_test_data[task] = v2_test_data.filter(lambda x: x['task'] == task, num_proc=8)
    print(f"Task: {task}, Number of samples: {len(task_specific_test_data[task])}")


Task: smol-property_prediction-lipo, Number of samples: 420
Task: smol-property_prediction-bbbp, Number of samples: 197
Task: smol-property_prediction-sider, Number of samples: 2860
Task: bace, Number of samples: 152
Task: forward_reaction_prediction, Number of samples: 1000
Task: chebi-20-mol2text, Number of samples: 3300
Task: smol-property_prediction-hiv, Number of samples: 4107
Task: reagent_prediction, Number of samples: 1000
Task: smol-property_prediction-clintox, Number of samples: 144
Task: qm9_homo, Number of samples: 684
Task: smol-property_prediction-esol, Number of samples: 112


In [29]:
task_specific_train_data = {}
for task in list_task:
    task_specific_train_data[task] = v2_train_data.filter(lambda x: x['task'] == task, num_proc=200)
    print(f"Task: {task}, Number of samples: {len(task_specific_train_data[task])}")

Task: smol-property_prediction-lipo, Number of samples: 3360
Task: smol-property_prediction-bbbp, Number of samples: 1569
Task: smol-property_prediction-sider, Number of samples: 22820
Task: bace, Number of samples: 1210
Task: forward_reaction_prediction, Number of samples: 107698
Task: chebi-20-mol2text, Number of samples: 4564
Task: smol-property_prediction-hiv, Number of samples: 32864
Task: reagent_prediction, Number of samples: 121896
Task: smol-property_prediction-clintox, Number of samples: 1144
Task: qm9_homo, Number of samples: 117660
Task: smol-property_prediction-esol, Number of samples: 888


In [30]:
check_keys = [
    'prompt_text',
    'target_text',
    '0-th_rejected_target_text'
]

In [31]:
for task in list_task:
    print("===" * 20)
    print(f"Task: {task}")
    for split in ['train', 'test']:
        print("===" * 20)
        print(f"Split: {split}")
        print("Number of samples: ", len(task_specific_train_data[task]) if split == 'train' else len(task_specific_test_data[task]))
        task_data = task_specific_train_data[task] if split == 'train' else task_specific_test_data[task]
        for key in check_keys:
            print(f"Key: {key}")
            print(task_data[key][0])
            print("\n")


Task: smol-property_prediction-lipo
Split: train
Number of samples:  3360
Key: prompt_text
<s>[INST] You are a helpful assistant for molecular chemistry, to address tasks including molecular property classification, molecular property regression, chemical reaction prediction, molecule captioning, molecule generation. 

<SELFIES> [N][C][=Branch1][C][=O][N][C][=C][Branch1][=C][C][=Branch1][C][=O][N][C@H1][C][C][C][N][C][Ring1][=Branch1][S][C][Branch1][=N][C][=C][C][=C][C][Branch1][C][F][=C][Ring1][#Branch1][F][=C][Ring2][Ring1][=Branch1] </SELFIES><GRAPH><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol><mol></GRAPH> What is the octanol/water distribution coefficient logD under the circumstance of pH 7.4 for the molecule given above? [/INST] 


Key: target_text
<FLOAT> <|+|><|1|><|.|><|3|><|2|><|0|><|0|> </FLOAT> </s>


Key: 0-th_rejected_target_text
<FLOAT> <|+|><|2|><|.|><|4|><|3|><